## 14_sql_task — Job task executing SQL logic (practice-job-5-weekly-sql)

Kept as a notebook task (spark.sql) so it works on BOTH old and new
workspaces with zero setup and zero cost (no SQL warehouse required).
The DAB bundle also ships a native sql_task variant (see bundle/src/sql).
Config: CATALOG/SCHEMA/PARTITION/CONFIG_PATH widgets -> setup_config() (C360-style)
PARTITION: all | latest | prev | YYYY-MM-DD (date guard lives in config_lib)


In [ ]:

%run ../lib/config_lib


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")
dbutils.widgets.text("PARTITION", "all", "all | latest | prev | YYYY-MM-DD")
dbutils.widgets.text("CONFIG_PATH", "", "Optional env config file (empty = built-in dev/stg/prd)")

CFG = setup_config(
    config_path=dbutils.widgets.get("CONFIG_PATH").strip(),
    run_date=dbutils.widgets.get("PARTITION").strip(),
    schema=dbutils.widgets.get("SCHEMA").strip(),
)


In [ ]:

src = CFG.table("practice_customers_enriched")
date = CFG.resolved_partition_date(src)
where_clause = "" if date is None else f"WHERE ingest_date = DATE '{date}'"

print(f"SQL filter: {where_clause or '(none — all partitions)'}")


In [ ]:

weekly = spark.sql(f"""
SELECT
    risk_category,
    segment,
    COUNT(*)                      AS customers,
    ROUND(AVG(annual_revenue), 2) AS avg_annual_revenue
FROM {src}
{where_clause}
GROUP BY risk_category, segment
ORDER BY risk_category, segment
""")

display(weekly)
print(f"WEEKLY SQL TASK DONE — risk x segment aggregation complete (PARTITION={CFG.run_date})")
